# Grover's Algorithm

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Arc
import numpy as np

def grover_image(show_oracle=False, show_grover=False):

    fig, ax = plt.subplots(figsize=(5, 5))

    # Axes
    ax.arrow(0, 0, 1.2, 0, head_width=0.04, head_length=0.06, fc='black', ec='black', linewidth=1)
    ax.arrow(0, 0, 0, 1.2, head_width=0.04, head_length=0.06, fc='black', ec='black', linewidth=1)
    ax.text(1.3, 0, r'$|\text{states}\perp s\rangle$', fontsize=12, ha='left')
    ax.text(-0.05, 1.25, r'$|s\rangle$', fontsize=12, ha='right')

    # Initial state
    r = 1.0
    theta_0 = np.pi / 18
    x_init = r * np.cos(theta_0)
    y_init = r * np.sin(theta_0)
    ax.arrow(0, 0, x_init, y_init, head_width=0.04, head_length=0.06, 
            fc='blue', ec='blue', linewidth=1)
    ax.text(x_init+0.1, y_init, r'$|+\rangle$', fontsize=12, ha='left')

    # Arc indicating the angle
    arc_radius = 0.4
    theta_1 = 0
    theta_2 = theta_0
    arc = Arc((0, 0), width=2*arc_radius, height=2*arc_radius,
                angle=0, theta1=np.degrees(theta_1), theta2=np.degrees(theta_2),
                color='blue', lw=1.5)
    ax.add_patch(arc)

    # Label the angle
    theta_mid = (theta_1+theta_2)/2
    mid_x = (arc_radius) * np.cos(theta_mid)
    mid_y = (arc_radius) * np.sin(theta_mid)
    ax.text(mid_x*1.1, mid_y*1.1, r'$\theta$', color='blue', fontsize=12, ha='left', va='center')

    # Oracle
    if (show_oracle):
        x_oracle = x_init
        y_oracle = -y_init
        ax.arrow(0, 0, x_oracle, y_oracle, head_width=0.04, head_length=0.06, 
                fc='white', ec='blue', linewidth=1, linestyle='-')
        ax.text(x_init+0.1, -y_init, r'$U_f\,|+\rangle$', fontsize=12, ha='left')

    # Reflection about |+>
    if (show_grover):
        theta_new = 3 * theta_0
        x_new = r * np.cos(theta_new)
        y_new = r * np.sin(theta_new)
        ax.arrow(0, 0, x_new, y_new, head_width=0.04, head_length=0.06, fc='green', ec='green', linewidth=1)
        ax.text(x_new+0.1, y_new, r'$G\,|+\rangle$', fontsize=12, ha='left')

        # Arc indicating the angle increment
        arc_radius = 0.6
        theta_1 = theta_0
        theta_2 = 3*theta_0
        arc = Arc((0, 0), width=2*arc_radius, height=2*arc_radius,
                    angle=0, theta1=np.degrees(theta_1), theta2=np.degrees(theta_2),
                    color='green', lw=1.5)
        ax.add_patch(arc)

        # Label the angle increment
        theta_mid = (theta_1+theta_2)/2
        mid_x = (arc_radius) * np.cos(theta_mid)
        mid_y = (arc_radius) * np.sin(theta_mid)
        ax.text(mid_x*1.1, mid_y*1.1, r'$2\theta$', color='green', fontsize=12, ha='left', va='center')

    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])

    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.show()

## Introduction and Oracle concept

Search problem, example:
- identify an entry in an unstructured database
- identify the solution of an NP-hard problem
- break a hash code


Possible data encoded on $n$ bits: $x\in\{0,1\}^n$

We have a function $f$ that identifies whether a given state $x$ is the solution $s$:

- $f(x) = 1 \quad$ if $x=s$
- $f(x) = 0 \quad$ otherwise

Such a function is called an "oracle".

On a quantum computer, possible solutions are encoded on $n$ qubits ($N = 2^n$ possible values):

\begin{equation*}
    \ket {0\cdots 00}, \quad \ket {0\cdots 01}, \quad \ket {0\cdots 10}, \quad \cdots, \quad \ket {1\cdots 11}
\end{equation*}

We use the oracle to build a quantum gate that can be inserted into a circuit:

\begin{equation*}
    U_f \ket{x} = (-1)^{f(x)} \ket{x}
\end{equation*}

This logic gate "recognizes" the solution of the problem described by $f$ and has the effect of flipping the sign of the vector component in the direction of the solution.

In practice, such a logic gate is implemented using an additional qubit (*ancilla*), whose sign we can flip if the oracle recognizes $x$ as a solution: $(f(x)=1)$. This operation is often written using an XOR operation:

\begin{equation*}
    \tilde U_f \ket{x}\ket{y} = \ket{x}\ket{y \oplus f(x)}
\end{equation*}

By initializing the qubit $\ket y$ with the state $\ket- = \frac{1}{\sqrt 2}(\ket 0 - \ket 1)$, it remains unchanged by the action of $\tilde U_f$ and we indeed construct a gate equivalent to $U_f$,

\begin{equation*}
    \tilde U_f \ket{x}\ket{-} = (-1)^{f(x)} \ket{x}\ket{-}
\end{equation*}

In general, the discussion of this type of algorithm pays little attention to the implementation of the oracle. However, since the quantum computing model generalizes the classical model, it is always theoretically possible to implement the computation of $f(x)$.

https://quantum.cloud.ibm.com/docs/en/api/qiskit/circuit_library#oracles

The advantage of the quantum computing model lies in the ability to evaluate in parallel all possible values of $x$ that can be encoded in state $\ket x$. This is achieved by initializing the qubits in state $\ket +$, a uniform superposition of all possible values, by applying a Hadamard gate to each qubit.

\begin{equation*}
    \ket{+} = H^{\otimes n} \ket{0 \cdots 00} = \frac{1}{\sqrt{N}} \ket{0 \cdots 00} + \frac{1}{\sqrt{N}} \ket{0 \cdots 01} + \cdots + \frac{1}{\sqrt{N}} \ket{s} + \cdots + \frac{1}{\sqrt{N}} \ket{1 \cdots 11}
\end{equation*}

Since this initial state is a uniform superposition of the $N=2^n$ possibilities, the probability of measuring this state in the solution state $\ket s$ is only $1/N$.

The action of the oracle only flips the sign of the coefficient of $\ket s$. Its role is merely to mark the component in the direction of the solution and, on its own, does not increase the probability of obtaining the solution state.

\begin{equation*}
    U_f \ket{+} = \frac{1}{\sqrt{N}} \ket{0 \cdots 00} + \frac{1}{\sqrt{N}} \ket{0 \cdots 01} + \cdots - \frac{1}{\sqrt{N}} \ket{s} + \cdots + \frac{1}{\sqrt{N}} \ket{1 \cdots 11}
\end{equation*}

## Geometric aspect

We can visualize the action of the oracle more simply by restricting the representation of the system to the plane spanned by the two vectors $\ket s$ and $\ket +$. Indeed, we can group the components of $\ket +$ perpendicular to $\ket s$ to simplify the notation:

\begin{equation*}
    \ket{+} = \frac{1}{\sqrt{N}} \ket{s} + \sqrt{1-\frac{1}{N}} \ket{x\perp s}
\end{equation*}

And reveal a geometric interpretation by rewriting the amplitudes with trigonometric functions:

\begin{equation*}
    \ket{+} = \sin(\theta) \ket{s} + \cos(\theta) \ket{x\perp s}
\end{equation*}

We then represent the situation on a graph where we place the solution state $\ket{s}$ (unknown) on the $y$-axis and the space of all other orthogonal states on the $x$-axis. The vector representing the initial state is tilted by a small angle $\theta \approx \sin(\theta) = 1/\sqrt{N}$ in the direction of the solution.

In [ ]:
grover_image(show_oracle=True)

The action of the oracle then consists of flipping the sign of this angle $\theta$:

\begin{equation*}
    U_f \ket{+} = -\sin(\theta) \ket{s} + \cos(\theta) \ket{x\perp s}
\end{equation*}

## Grover iteration

As mentioned above, the oracle does not increase the probability of measuring our quantum state in the solution state. The probability remains the same as at the start:

\begin{equation*}
    |\langle s|(U_f|+\rangle)|^2 = \sin^2(\theta) = \frac 1N = |\braket{s|+}|^2
\end{equation*}

To exploit the action of the oracle and increase the probability of obtaining the solution $\ket s$, we perform a geometric reflection operation around the initial state $\ket{+}$, which brings our quantum state closer to $\ket{s}$.

The successive application of the oracle $U_f$ and this reflection around $\ket{+}$ constitutes a complete iteration of Grover's algorithm, which we represent by the operator $G$.

In [ ]:
grover_image(show_oracle=True, show_grover=True)

Algebraically, this operation is represented by the operator

\begin{equation*}
    U_+ = 2|+\rangle \langle+| - I
\end{equation*}

Its action on a state $\ket\psi$ indeed flips the sign of the component perpendicular to $\ket +$:

\begin{equation*}
    \ket\psi = \alpha \ket{+} + \beta \ket{\psi_{\perp}}
\end{equation*}

\begin{align*}
    U_+ \ket\psi 
    &= 2|+\rangle \langle+ \ket\psi - I \ket\psi \\
    &= 2\alpha \ket{+} - (\alpha \ket{+} + \beta \ket{\psi_{\perp}}) \\
    &= \alpha \ket{+} - \beta \ket{\psi_{\perp}}
\end{align*}

Successive applications of the Grover iteration each rotate the vector by an angle $2\theta$. The probability of measuring the solution state therefore increases progressively. After $k$ iterations, it equals

\begin{equation*}
    |\braket{s|\psi}|^2 = \sin^2((2k+1)\theta) \qquad k = 0,1,2,\cdots
\end{equation*}

It is maximal for $k \approx \pi/4\theta$ iterations. Grover's algorithm thus allows us to complete the search problem with only $O(\sqrt{N})$ oracle calls. (But beware, beyond the optimal number of iterations, the probability starts to decrease!)

![title](../figures/grover_full.png)

![title](../figures/grover_iter.png)

## Oracle that identifies a particular state

In [ ]:
from qiskit.quantum_info import Operator
from qiskit import QuantumCircuit
import numpy as np

def oracle(bitstring):

    n = len(bitstring)
    oracle_matrix = np.eye(2**n)
    index = int(bitstring, 2)
    oracle_matrix[index, index] = -1
    return Operator(oracle_matrix)

In [ ]:
etat_reconnu = "0010"

Uf = oracle(etat_reconnu)
display(Uf.draw("latex"))

n = len(etat_reconnu)
circuit_oracle = QuantumCircuit(n)
circuit_oracle.unitary(Uf, circuit_oracle.qubits, label='Uf')
circuit_oracle.draw('mpl')

## Circuit with Grover iterations

On a quantum computer, $U_+$ is implemented using Hadamard gates,

\begin{align*}
    U_+ 
    &= 2|+\rangle \langle+| - I \\
    &= 2 H^{\otimes n} |0\rangle \langle0| H^{\otimes n} - H^{\otimes n} H^{\otimes n} \\
    &= H^{\otimes n} (2|0\rangle \langle0| - I) H^{\otimes n} \\
    &= H^{\otimes n} U_0 H^{\otimes n}
\end{align*}

which allow performing the geometric reflection around the state $\ket{0\cdots 00}$, which can be implemented using a controlled NOT gate.

(Suggested exercise: show that the implementation of $U_0$ below indeed flips the sign of components different from $\ket{0\cdots 00}$)

In [ ]:
# First, apply the oracle
circuit_grover_iter = circuit_oracle.copy()

# Then, apply the Hadamard gates
circuit_grover_iter.h(range(n))

# Then, apply the geometric reflection around state |0...00>
circuit_grover_iter.x(range(n))
circuit_grover_iter.h(0)
circuit_grover_iter.mcx(list(range(1, n)),0)  # Controlled NOT gate on qubit 0
circuit_grover_iter.h(0)
circuit_grover_iter.x(range(n))

# Finally, apply the Hadamard gates again
circuit_grover_iter.h(range(n))

circuit_grover_iter.draw('mpl')

In [ ]:
#from qiskit.circuit.library import grover_operator

#circuit_grover_iter = grover_operator(circuit_oracle)
#circuit_grover_iter.draw('mpl')
#circuit_grover_iter.decompose().draw('mpl')

In [ ]:
circuit = QuantumCircuit(n)

# Initial superposition (|+> state)
circuit.h(range(n))

# Grover iterations
#num_iters = 1
num_iters = np.floor((np.pi/4)*np.sqrt(2**n))

for _ in range(int(num_iters)):
    circuit.append(circuit_grover_iter.to_gate(), range(n))

circuit.measure_all()
circuit.draw('mpl')

In [ ]:
# from qiskit_ibm_runtime import QiskitRuntimeService
# service = QiskitRuntimeService(channel="ibm_quantum_platform", token="G9lfwEwpvKBJf53tgvLt-uMINdNb7LaXAh6_iq39mFiM")
# backend = service.least_busy(simulator=False, operational=True)

### Too much noise: one Grover iteration is better in this case
#from qiskit_ibm_runtime.fake_provider import FakeManilaV2
#backend = FakeManilaV2()

from qiskit_aer import AerSimulator
backend = AerSimulator()

In [ ]:
from qiskit.compiler import transpile

circuit_transpiled = transpile(circuit, backend, optimization_level=3)
circuit_transpiled.draw('mpl')

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler
sampler = Sampler(backend)

job = sampler.run([circuit_transpiled], shots=1000)

job_id = job.job_id()
print("Job id: ", job_id)

In [ ]:
#job = service.job(job_id)
result = job.result()

from qiskit.visualization import plot_histogram
plot_histogram(result[0].data.meas.get_counts())